In [2]:
import train
%load_ext autoreload
%autoreload 2
import tools
from analysis import performance
from analysis import standard_analysis
from analysis import clustering
from analysis import variance
from analysis import taskset
from analysis import varyhp
from analysis import data_analysis
from analysis import contextdm_analysis
from analysis import posttrain_analysis
from network import Model
import tensorflow as tf
import numpy as np
import random as random
import os
import matplotlib.pyplot as plt

/home/fbraidi/miniconda3/envs/thesis-env/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/fbraidi/miniconda3/envs/thesis-env/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/fbraidi/miniconda3/envs/thesis-env/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/fbraidi/miniconda3/env

In [20]:
model_dir='./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024'
replace_rule= ['fdgo', 'reactgo', 'delaygo', 'fdanti', 'reactanti', 'delayanti',
              'dm1', 'dm2', 'contextdm1', 'contextdm2', 'multidm',
              'delaydm1', 'delaydm2', 'contextdelaydm1', 'contextdelaydm2', 'multidelaydm',
              'dmsgo', 'dmsnogo', 'dmcgo', 'dmcnogo', 'random', 'random_mod']

compositional_rules={'go':[-1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                     'dm1':[0,0,0,1,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                     'dm2':[0,0,0,1,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0],
                     'contextdm1':[0,0,0,1,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0],
                     'contextdm2':[0,0,0,1,0,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0],
                     'multidm':[0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0]}

pure_tasks_perfs=[]
comp_tasks_perf=[]

for i in range(len(replace_rule)):

    rule_strength=[0]*len(replace_rule)
    rule_strength[i]=1
    perf, _ = taskset.run_network_replacerule(model_dir, replace_rule[i], replace_rule, rule_strength)
    pure_tasks_perfs.append(perf)

for el in compositional_rules:

    perf, _ = taskset.run_network_replacerule(model_dir, 'delayanti', replace_rule, compositional_rules[el])
    comp_tasks_perf.append((el,perf))

for i in range(len(replace_rule)):

    print(f'Performance on task {replace_rule[i]}: ', pure_tasks_perfs[i])

for (fam,perf) in comp_tasks_perf:

    print(f'Performance on delayanti when instructed with {fam} family: ', perf)

INFO:tensorflow:Restoring parameters from ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt
Model restored from file: ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt
INFO:tensorflow:Restoring parameters from ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt
Model restored from file: ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt
INFO:tensorflow:Restoring parameters from ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt
Model restored from file: ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_c

In [4]:
model_dir='./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024'
replace_rule= ['fdgo', 'reactgo', 'delaygo', 'fdanti', 'reactanti', 'delayanti',
              'dm1', 'dm2', 'contextdm1', 'contextdm2', 'multidm',
              'delaydm1', 'delaydm2', 'contextdelaydm1', 'contextdelaydm2', 'multidelaydm',
              'dmsgo', 'dmsnogo', 'dmcgo', 'dmcnogo', 'random', 'random_mod']

out_of_diag_perc=0.002
grid=np.linspace(0,1,11)

performances=np.zeros(shape=(11,11,11,11,11,11))

for i,a in enumerate(grid):
    for j,b in enumerate(grid):
        for k,c in enumerate(grid):
            for l,d in enumerate(grid):
                for m,e in enumerate(grid):
                    for n,f in enumerate(grid):
            
                        tot=(int((a+b+c+d+e+f)*10))/10

                        if tot < 0.8 or tot > 1.2:
                            if np.random.rand() < 1-out_of_diag_perc:
                                continue    

                        print(a,b,c,d,e,f)

                        rule_strength= a*np.array([-1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0])+\
                                        b*np.array([0,0,0,1,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0])+\
                                        c*np.array([0,0,0,1,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0,0])+\
                                        d*np.array([0,0,0,1,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0,0])+\
                                        e*np.array([0,0,0,1,0,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0,0])+\
                                        f*np.array([0,0,0,1,0,0,0,0,0,0,-1,0,0,0,0,1,0,0,0,0,0,0])

                        perf, _ = taskset.run_network_replacerule(model_dir, 'delayanti', replace_rule, rule_strength)
                        performances[i,j,k,l,m,n]=perf

np.save(os.path.join(model_dir,'comp_rule_linear_comb_2.npy'),performances)
        

0.0 0.0 0.0 0.0 0.0 0.8
Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Colocations handled automatically by placer.


Instructions for updating:
Use standard file APIs to check for files with this prefix.
INFO:tensorflow:Restoring parameters from ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt


2025-03-18 17:18:57.745364: I tensorflow/core/platform/cpu_feature_guard.cc:141] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 AVX512F FMA
2025-03-18 17:18:57.754625: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 3099855000 Hz
2025-03-18 17:18:57.760077: I tensorflow/compiler/xla/service/service.cc:150] XLA service 0x4c57dd0 executing computations on platform Host. Devices:
2025-03-18 17:18:57.760099: I tensorflow/compiler/xla/service/service.cc:158]   StreamExecutor device (0): <undefined>, <undefined>


Model restored from file: ./../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/model.ckpt


KeyboardInterrupt: 

In [5]:
loaded=np.load(os.path.join(model_dir,'comp_rule_linear_comb_2.npy'),allow_pickle=True)

plt.imshow(loaded)

FileNotFoundError: [Errno 2] No such file or directory: './../models/fdanti_delaygo_fdgo_delaydm1_dm1_delaydm2_dm2_contextdelaydm1_contextdm1_contextdelaydm2_dm2_multidelaydm_multidm_1024/comp_rule_linear_comb_2.npy'